In [ ]:
%pip install psutil numpy

In [ ]:
import time
import psutil
import numpy as np

# Performance Measurement in Python: What Are We Actually Measuring? 

Measuring performance in scientific and technical computing is more difficult than it first appears. Modern programs interact with the operating system, hardware, numerical libraries, and sometimes accelerators such as GPUs. Because of this, common terms like “runtime,” “CPU usage,” or “efficiency” can mean different things in different situations. A program may be slow because it is doing a lot of computation, because it is waiting for input or output, or because work is happening in parallel or on another device. Simple timing measurements often hide these differences and can therefore be misleading. 

The goal of this notebook is to introduce a small set of practical measurement tools and patterns, and to help develop correct mental models of what these measurements actually represent. The focus is not on optimization, but on interpretation: learning how to choose appropriate metrics, how to read them correctly, and how to recognize when performance results should be treated with caution.
 


---


## CPU Time vs Wall Time


**Wall time** is what a user experiences; it measures the passage of time as seen on a clock "on your wall".
**CPU time** is how much time the CPU spent executing your process vs waiting on other resources.

They are often not the same, especially when waiting, sleeping, or doing IO. 

Measuring these times uses a simple pattern: the "Timing Interval":

```python
start_time = measure_time()         # get t0
do_work()                           
end_time = measure_time()           # get t1
total_time = end_time - start_time  # interval = t1 - t0
```


| Code  | Description |
| :-- | :-- |
| `time.process_time()` | CPU time used by the current process measurement |
| `time.perf_counter()` | wall-clock time measurement |
| `for _ in range(10): do_work()` | Run the `do_work()` function 10 times. |
| `total_process_time / total_wall_time` | Rough CPU utilization estimate |

### Exercises

The exercises below focus on measuring the following functions:

In [ ]:
def sum_of_ints(n):
    """A CPU-Heavy, Memory-Light, Python-only task."""
    total = 0
    for i in range(1, n + 1):
        total += i
    return total


def wait(secs):
    """A CPU-Light, Memory-Light task that relies on OS system calls."""
    import time
    time.sleep(secs)


**Example**: What do we expect, and what do we measure?  Using our function, how Much CPU time does it take to compute the sum of the first 10 million integers (i.e. `sum_of_ints(10_000_000)`)?  

In [ ]:
start_cpu = time.process_time()
sum_of_ints(10_000_000)
cpu = time.process_time() - start_cpu
cpu

**Exercise**: What do we expect, and what do we measure?  Using our function, how Much CPU time does it take to compute the sum of the first 20 million integers?  

Is the result roughly double the 10 million integers? Also, how stable is this measurement; do you get the same result every time you run the code?

**Exercise**: What do we expect, and what do we measure?  Using our function, how Much CPU time does it take to compute the sum of the first 1000 integers?  

Note: There will likely be something strange about the result. To get a clearer measurement on this "micro-benchmark", you may have to average the time over many runs.

**Exercise**: How much real time (i.e. "wall time") passed when calculating the sum of ten million integers?  How does this compare to the process time?

**Exercise**: How much real time (i.e. "wall time") passes when having the computer wait half a second (i.e. `wait(0.5)`)?  How does this compare to the process time?

**Exercise**: The `timeit` package is a very helpful way to auto-repeat micro-benchmarks, to get failry-accurate runs.  It's quite convenient, as it can be run as a function, a line magick in a notebook (to run a single line), and a cell magick (to run everyhting in a cell).

To see it work, go ahead and run the cells below, which test the `sum_of_ints()` function with a very small value.

In [ ]:
from timeit import timeit
timeit('sum_of_ints(100)', globals=globals())  # result is in microseconds

In [ ]:
%timeit sum_of_ints(100)

In [ ]:
%%timeit
sum_of_ints(100)

**Final Thoughts**: Wall Time measurements, though fickle, are very revealing about performance, but to start getting an idea of where this time is going, one needs to compare both process and wall times. 

---



## IO vs CPU: Waiting Is Not Working

IO-bound work spends most of its time **waiting**, not computing. Wall time increases, but CPU time barely moves, either because it cannot proceed until the data transfer is complete, or because it is waiting for some other condition to be met.

**I/O ("input/output")** refers to any situation where a program must wait for data to move between the CPU and something outside the CPU itself (e.g. Hard Disk, Network, GPU, but not including RAM). This includes reading data from disk, writing results to files, loading datasets over the network, saving model checkpoints, logging outputs, or communicating with other processes or services. 

For many real-world workflows in machine learning and scientific computing, I/O is a substantial part of the total runtime, even if it is less visible than numerical computation. How I/O is done--number of requests, batch sizes, total data transferred, caching, etc, all can be tuned to change performance.

The more So, how can we see what an algorithm is doing?  In this section, we'll use the [psutil](https://pypi.org/project/psutil/) package to measure disk reading and writing, to help us distinguish between **read/write** bottlenecks and **waiting** bottlenecks.


| Code | Description |
| :-- | :-- |
| `psutil.Process()`   | Handle to current process |
| `io = proc.io_counters()` | Disk read/write counters  |
| `io.read_count`  | Number of read operations performed by the process (e.g. file reads), regardless of size |
| `io.read_bytes`  | Total number of bytes read from storage by the process  |
| `io.write_count` | Number of write operations performed by the process (e.g. file writes) |
| `io.write_bytes` | Total number of bytes written to storage by the process |


### Exercises


The following exercises benchmark the following functions:

In [ ]:
# Creates a data file for the functions to use.

def wait(n: int):
    time.sleep(n)

def write_integers(n: int):
    with open('integers.dat', 'wb') as f:
        for i in range(1, 500_000 + 1):
            f.write(i.to_bytes(4, byteorder='little', signed=False))  # write as int32s)

def read_integers(n):
    np.fromfile('integers.dat', dtype=np.int32, count=n)




**Exercise**: `wait(1)`: Measure I/O counters before and after calling `wait(1)`, and record their differences.  Which Counters changed?





**Exercise**: `write_integers(1000)`: Measure I/O counters before and after calling `write_integers(1000)`, and record their differences.  Which Counters changed?





**Exercise**: `read_integers(1000)`: Measure I/O counters before and after calling `read_integers(1000)`, and record their differences.  Which Counters changed?



## Optimizing through Profiling with `line_profiler`

**Optimization without Profiling is worthless**.  To really know where to make changes, we need to measure them, and that's where profiling comes in.  For microbenchmarking and small experiments, `line_profiler` is a great way to build a sense of small bits of  code's performance. 

It will tell you, from highest-priority to lowest:
  1. **Where the biggest impact can be had**: How much total time was spent running a given line.
  2. **If the code is "hot", run many times**: How many times a line was called.
  3. **If the code itself is hiding a lot of work**: How much time it usually takes to run a given line.


| Code | Description |
| :-- | :-- |
| `%load_ext line_profiler` | Loads/Imports the line_profiler tool |
| `prof = %lprun -r -f <function_to_profile> <code_to_run>` | Run the profiler and return the results |
| `prof.print_stats()` | print the profiling stats |


### Exercises



**Exercise**: 

Code that has low "**Effective CPU Utilization (process time / wall time)**" is often an indicator of "**IO-Dominated Code**".  Optimization of IO is very different than optimization of CPU, and so code that weaves both together is trickier than code that seperates the two.

Below is are two alternative versions of "sum_of_ints", which reads the first **N* values from a pre-made file and calculates their sum.  One weaves them together, and another seperates them. We'll use these functions for our optimization work.

In [ ]:
# Creates a data file for the functions to use.
with open('integers.dat', 'wb') as f:
    for i in range(1, 10_000_000 + 1):
        f.write(i.to_bytes(4, byteorder='little', signed=False))  # write as int32s)


# The functions
def sum_of_ints2(n: int) -> int:
    total = 0
    with open('integers.dat', 'rb') as f:  
        for _ in range(1, n + 1):
            dat = f.read(4)   # IO Focus
            value = int.from_bytes(dat, byteorder='little', signed=False)  
            total += value  # CPU Focus
    return total


def sum_of_ints3(n: int) -> int:

    # IO Focus
    packets = []
    with open('integers.dat', 'rb') as f:
        for _ in range(n):
            dat = f.read(4)
            packets.append(dat)
    data = b"".join(packets)
    
    # CPU Focus
    total = 0
    for start_idx in range(0, len(data), 4):
        value = int.from_bytes(data[start_idx:start_idx+4], byteorder='little', signed=False)
        total += value


    return total




In [ ]:
%load_ext line_profiler

**Exercise**: Which of the two functions is fastest at calculating the sum of the first 10,000 ints?

**Exercise**: Use line profiler to inspect `sum_of_ints3()`.  Which section in this function is the computer spending the most time: the IO section, or the CPU section?

Answer: ___________

**Exercise**: Refactor `sum_of_ints3()`, using the information in the line profiler to guide the process, aiming for a >50% speedup.

## GPU Time

GPUs execute work **asynchronously** with respect to the CPU. When a Python program launches a GPU operation, the call often returns immediately, while the actual computation continues in the background on the device. This means that timing using standard wall-clock functions can significantly underestimate GPU execution time, because the CPU may stop the clock before the GPU has finished its work.

To measure GPU wall time correctly, we must explicitly synchronize the CPU with the GPU before starting and after finishing the timed section. 

In this section, we use two small helper functions: one to check whether NVIDIA’s management library (NVML) is available, and another to perform a blocking GPU synchronization using PyTorch when possible. This ensures that the measured time reflects the full duration of the GPU computation, not just the time required to launch it.

| Code | Description  |
| :-- | :-- |
| `has_nvml()`   | Checks whether the NVIDIA Management Library (NVML) is installed and can be initialized successfully |
| `pynvml.nvmlInit()`  | Initializes NVML and establishes communication with the NVIDIA driver     |
| `pynvml.nvmlShutdown()` | Cleans up NVML resources after the check   |   |
| `torch.cuda.is_available()` | Indicates whether PyTorch can access a CUDA-capable GPU     |
| `torch.cuda.synchronize()`  | Blocks the CPU until all queued GPU work has completed |


### Utils

In [ ]:
def has_nvml() -> bool:
    """Returns True, if NVIDIA Management Library is installed and works fine"""
    try: 
        import pynvml
        pynvml.nvmlInit()
        pynvml.nvmlShutdown()
        return True
    except Exception:
        return False
    

def gpu_synchronize():
    """Returns True, if PyTorch could call a blocking synchronization command on the GPU."""
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            return True
    except ImportError:
        pass


# 1. Synchronize GPU and Start the Clock
gpu_synchronize()
start = time.perf_counter()

# 2. Run the task
sum_of_ints(1000)  

# 3. Synchronize the GPU again to be sure it's finished, then stop the clock.
gpu_synchronize()
gpu_wall_time = time.perf_counter() - start
gpu_wall_time

**Exercise**: Update the code above to test out a GPU-using function.